# Python. Семинар 4. Ссылки вблизи


## 1. Умножение списка повторяет ссылку, а не содержимое

Таблицу из нулей хочется построить умножением — запись короткая и выглядит
понятно:


In [ ]:
grid = [[0] * 3] * 3

grid

Выглядит правильно. Изменим одну клетку:


In [ ]:
grid[0][0] = 1

grid

Единица появилась во всех трёх строках: `[[0] * 3] * 3` не копирует внутренний
список, а трижды записывает **ссылку** на один и тот же объект.


In [ ]:
grid[0] is grid[1], grid[0] is grid[2]

Внутреннее `[0] * 3` при этом безопасно: нули неизменяемы. Опасно повторение
изменяемого объекта.

Чтобы строки были разными объектами, каждую создают отдельно — тело включения
выполняется на каждом шаге заново:


In [ ]:
grid = [[0] * 3 for _ in range(3)]
grid[0][0] = 1

grid, grid[0] is grid[1]

## 2. `+=` и `+` для списка — это разные операции

Для чисел `a += 1` и `a = a + 1` неразличимы. Для списка разница принципиальна.
Заведём второе имя для того же списка и применим `+=`:


In [ ]:
values = [1, 2]
alias = values

values += [3]

values, alias, values is alias

Изменился сам объект, поэтому изменение видно и через второе имя. Теперь то же
самое через `+`:


In [ ]:
values = [1, 2]
alias = values

values = values + [3]

values, alias, values is alias

`+` собрал новый список и привязал к нему имя `values`, а `alias` остался у
старого объекта. То есть `+=` для списка — это `extend`, а `+` — копия.

Для неизменяемых типов выбора нет: менять объект нельзя, поэтому `+=` тоже
создаёт новый.


In [ ]:
point = (1, 2)
same = point

point += (3,)

point, same, point is same

## 3. `a[:] = ...` меняет содержимое, не меняя объект

Иногда нужно заменить содержимое списка целиком, но сохранить сам объект —
например, потому что на него ссылается кто-то ещё. Обычное присваивание для
этого не годится: оно перепривязывает имя.


In [ ]:
values = [3, 1, 2]
alias = values

values = sorted(values)

values, alias, values is alias

`alias` по-прежнему смотрит на старый неотсортированный список. Присваивание
**срезу** работает иначе: слева не имя, а содержимое объекта.


In [ ]:
values = [3, 1, 2]
alias = values

values[:] = sorted(values)

values, alias, values is alias

Тот же приём убирает элементы, не создавая нового списка, — второй ответ на
задачу прошлого семинара про удаление во время перебора:


In [ ]:
values = [1, 2, 3, 4]
alias = values

values[:] = [x for x in values if x % 2 != 0]

values, alias is values

## 4. `sort` меняет список, `sorted` строит новый

`sorted` из лекции 3 возвращает новый список и не трогает исходный. У списка
есть и собственный метод `sort`, который сортирует на месте — и **ничего не
возвращает**:


In [ ]:
values = [3, 1, 2]
result = values.sort()

values, result

Отсюда частая ошибка: `values = values.sort()`. Список отсортировался, но имени
`values` присвоили `None` — результат потерян.


In [ ]:
values = [3, 1, 2]
values = values.sort()

values

Правило простое: если функция меняет объект на месте, она возвращает `None`, и
её результат присваивать нельзя. Так устроены `list.sort`, `list.append`,
`list.reverse` и `set.add`. Если нужен новый объект — берите `sorted` или
срез `[::-1]`.


## 5. Строка, которая падает с ошибкой и всё-таки срабатывает

В лекции разбиралось, что список внутри кортежа изменить можно, а сам кортеж —
нельзя. Соединим оба факта в одной строке. Возьмём кортеж, единственный элемент
которого — пустой список, и допишем в этот список элемент через `+=`.


In [ ]:
box = ([],)

box[0] += [1]

Ячейка завершилась ошибкой `TypeError: 'tuple' object does not support item
assignment` — кортеж не даёт присвоить свой элемент. Посмотрим теперь, что
внутри кортежа:


In [ ]:
box

Единица на месте: операция и упала, и выполнилась. `box[0] += [1]` — это два
шага подряд:

1. `box[0].__iadd__([1])` — список меняет себя на месте и возвращает себя же;
2. `box[0] = <результат>` — запись результата обратно в кортеж.

Первый проходит, второй падает. У `append` такой проблемы нет: он ничего не
присваивает.


In [ ]:
box = ([],)
box[0].append(1)

box

Практический вывод: `+=` для элемента кортежа писать не нужно никогда. А если
кортеж должен быть по-настоящему неизменяемым, внутри у него не должно быть
изменяемых объектов.


## 6. `id` уникален только среди живых объектов

`id` в лекции служил способом отличить объекты друг от друга. У него есть
оговорка: число уникально только пока объект жив. Как только объект больше
никому не нужен, память освобождается — и следующий объект может занять тот же
адрес.


In [ ]:
id([]) == id([])

Первый список создали, взяли `id` — и он стал не нужен: память освободилась и
досталась второму. Числа совпали, хотя объекты разные и одновременно не
существовали.

Если оба списка держать в переменных, адреса различаются:


In [ ]:
first = []
second = []

id(first) == id(second), first is second, first == second

Сравнивать сохранённые ранее числа `id` нельзя — они годятся только для живых
объектов. Вместо `id(a) == id(b)` пишут `a is b`.
